# Week 4 — Predictive Modeling and Optimization
Forecast delivery time, evaluate candidate models, and test a simple transport-mode optimization scenario.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

df = pd.read_csv('../data/raw/week3_week4_logistics_simulated.csv')


In [ ]:
features = ['zone','transport_mode','shipment_volume','distance_km','traffic_index','weather_index','warehouse_load_pct','vehicle_utilization_pct']
target = 'delivery_time_hours'
X, y = df[features], df[target]
cat = ['zone','transport_mode']
num = [c for c in features if c not in cat]
pre = ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore'), cat), ('num','passthrough',num)])


In [ ]:
model = Pipeline([('preprocess', pre), ('model', RandomForestRegressor(n_estimators=250, random_state=42, min_samples_leaf=2))])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2, random_state=42)
model.fit(X_train,y_train)
pred = model.predict(X_test)
print('MAE', mean_absolute_error(y_test,pred))
print('RMSE', np.sqrt(mean_squared_error(y_test,pred)))
print('R2', r2_score(y_test,pred))


In [ ]:
scenario = df.copy()
mask = (scenario.distance_km > df.distance_km.median()) & (scenario.traffic_index > df.traffic_index.median()) & (scenario.transport_mode == 'Bike')
scenario.loc[mask,'transport_mode'] = 'Van'
print('Affected shipments:', int(mask.sum()))
